# Определение количества дикторов

In [16]:
import sys
import os
from pathlib import Path
import pandas as pd
import torch
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pyannote.audio import Pipeline
import warnings
warnings.filterwarnings("ignore")

load_dotenv()

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config

DEVICE = 'cpu'

In [17]:
HF_TOKEN = os.getenv('HF_TOKEN')

pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    token=HF_TOKEN,
)
pipeline.to(torch.device(DEVICE))

In [23]:
def count_speakers(path: str) -> int:
    result = pipeline(path)
    annotation = result.speaker_diarization
    return len(annotation.labels())

# Быстрый тест на одном файле
test_path = str(next(config.GOOD_DIR.glob('*.wav')))
print(f'Тест: {Path(test_path).name}: {count_speakers(test_path)} спикер(а)')

Тест: eu.6ca2e8f3-e4bd-44a0-8f6e-b13bb20f3fb0.wav: 1 спикер(а)


In [19]:
df = pd.read_csv(config.DATASET_CSV, encoding='utf-8')
df['_path'] = df.apply(lambda r: str(config.DATA_DIR / r['dir'] / r['filename']), axis=1)

if 'n_speakers' not in df.columns:
    df['n_speakers'] = pd.NA
else:
    df['n_speakers'] = pd.to_numeric(df['n_speakers'], errors='coerce')

need_idx = df[df['n_speakers'].isna() | (df['n_speakers'] == -1)].index.tolist()
print(f'Записей в датасете:  {len(df)}')
print(f'Требуют обработки:   {len(need_idx)}')
print(f'Уже обработано:      {len(df) - len(need_idx)}')

errors = []
for i, idx in enumerate(need_idx):
    path = df.at[idx, '_path']
    try:
        n = count_speakers(path)
    except Exception as e:
        n = -1
        errors.append(f"{df.at[idx, 'filename']}: {e}")
    df.at[idx, 'n_speakers'] = n

    if (i + 1) % 100 == 0 or (i + 1) == len(need_idx):
        df.drop(columns=['_path']).to_csv(config.DATASET_CSV, index=False, encoding='utf-8')
        print(f'[{i+1}/{len(need_idx)}] промежуточное сохранение, ошибок: {len(errors)}')

print(f'Ошибок: {len(errors)}')
for e in errors:
    print(f'{e}')

Записей в датасете:  2772
Требуют обработки:   0
Уже обработано:      2772
Ошибок: 0


In [20]:
df_out = df.drop(columns=['_path'], errors='ignore')
df_out['n_speakers'] = df_out['n_speakers'].astype('Int64')
df_out.to_csv(config.DATASET_CSV, index=False, encoding='utf-8')
print(f'Сохранено: {config.DATASET_CSV}')
print(f'Записей: {len(df_out)}')
df_out[['filename', 'label', 'duration', 'n_speakers']].sample(5)

Сохранено: /Users/dk/Desktop/ВШЭ/ВКР/HSE_VKR_DetectingSpeechDefects/data/dataset.csv
Записей: 2772


,filename,label,duration,n_speakers
2022,eu.2736b15b-9f42-404c-ae91-9d17c78dbd45.wav,1,10.36000,1
2073,eu.34f0d1f0-e326-43f4-8c1b-85724fafdcf4.wav,1,23.82325,1
2080,eu.35d54683-0cf5-46ac-b1ce-f2565d7d2011.wav,1,6.28000,1
2752,eu.faa0fd81-9695-457f-8351-0d5c3990a284.wav,1,18.58000,1
1072,eu.99726fea-900e-4716-a1f5-e443b513e341.wav,0,8.24000,1
